In [1]:
import oqupy
import oqupy.operators as op
import numpy as np
from oqupy.iTEBD_TEMPO_useoqupybath import iTEBD_TEMPO_oqupy
from oqupy.process_tensor import TTInvariantProcessTensor
from oqupy.tti_tempo import TTITempo
from oqupy.tti_tempo import TTITempoCounting
import matplotlib.pyplot as plt


from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import minimize,Bounds

pt_parameters = {'epsrel':10**(-7),
                 'alpha':0.1,
                 'omega_cutoff':1,                 
                 'temp':0.131,
                 'dt':0.25}

omega_cutoff = pt_parameters['omega_cutoff']
alpha = pt_parameters['alpha']
temperature = pt_parameters['temp']
epsrel = pt_parameters['epsrel']
# dt = 1./omega_cutoff/np.sqrt(3)
dt=pt_parameters['dt']

tcut=27

Rho_0=oqupy.operators.spin_dm('x+')

# spectral density (without cutoff)
def j(w):
    return 2*alpha*w


In [2]:
correlations = oqupy.PowerLawSD(alpha=alpha,
                                zeta=1,
                                cutoff=omega_cutoff,
                                cutoff_type='exponential',
                                temperature=temperature)
bath = oqupy.Bath(op.sigma("z")/2.0, correlations)
parameters=oqupy.TempoParameters(dt=dt,epsrel=epsrel,dkmax=int(tcut/dt)+1)

In [3]:
u=0.01
correlationscf=oqupy.bath_correlations.CustomCountingSD(j_function=j,cutoff=omega_cutoff,u=u,
                                                 cutoff_type='exponential',temperature=temperature)

bathcf = oqupy.Bath(op.sigma("z")/2.0, correlationscf)

In [4]:
# converged parameters for different protocol times 
protocol_times=[10,50,70,100,200]

import dill

with open('processtensor_simplemodel', 'rb') as f:
    # The protocol version used is detected automatically, so we do not
    # have to specify it.
    processtensor = dill.load(f)

with open('processtensorCF_simplemodel', 'rb') as f:
    # The protocol version used is detected automatically, so we do not
    # have to specify it.
    processtensorcf = dill.load(f)


In [5]:
def discrete_hamiltonian(hx):
    return hx*op.sigma('x')/2
system = oqupy.ParameterizedSystem(discrete_hamiltonian)

target_derivative = -1j*np.identity(2)/u

In [6]:
# Cost Function
num_params=1

def heatandgrad(paras,process_tensor,num_steps):
    """""
    Take a numpy array [hx0, hz0, hx1, hz1, ...] over full timesteps and
    return the fidelity and gradient of the fidelity to the global target_derivative
    """

    # Reshape flat parameter list to form accepted by state_gradient: [[hx0,hz0],[hx1,hz1,]...]
    reshapedparas = [i for i in (paras.reshape((-1,num_params))).tolist() for j in range(2)]
    #parameter_list=np.array([itemopt_steps=[9] for pair in zip(paras[:num_steps],paras[num_steps:2*num_steps]) for item in pair])
    #reshapedparas = [i for i in (parameter_list.reshape((-1,num_params))).tolist() for j in range(2)]
    reshapedparas = np.array(reshapedparas)

    gradient_dict = oqupy.state_gradient(
        system=system,
        initial_state=Rho_0,
        target_derivative=target_derivative,
        process_tensors=[process_tensor],
        parameters=reshapedparas,
        num_steps=num_steps,
        progress_type='silent')
    
    fs=gradient_dict['final_state']
    gps=gradient_dict['gradient']

    heat=np.trace(fs).imag/u

    # Adding adjacent elements
    for i in range(0,gps.shape[0],2): 
        gps[i,:]=gps[i,:]+gps[i+1,:]
        
    gps=gps[0::2]

    x=[]
    for i in range(0,gps.shape[1]): 
        x.append(gps[:,i])
    
    gps=np.array(x)

    # Return the minus the gradient as infidelity is being minimized 
    return heat,(1.0*gps.reshape((-1)).real).tolist()

In [8]:
import time

opt_dict={protocol_times[0]:[],
                 protocol_times[1]:[], 
                 protocol_times[2]:[],
                 protocol_times[3]:[],
                 protocol_times[4]:[]}

min_heats=[]
opt_runtimes=[
]

for t_prot in [50]:
    num_steps=int(t_prot/processtensor.dt)
    hx=np.zeros(num_steps)
    parameter_list=[item for pair in zip(hx) for item in pair]
    start = time.time()
    
    optimization_result = minimize(
                            fun=heatandgrad,
                            x0=parameter_list,
                            args=(processtensorcf,num_steps),
                            method='l-bfgs-b',
                            jac=True,
                            options = {'disp':True, 'gtol': 7e-04}
    )
    end = time.time()
    opt_runtimes.append(end-start)

    opt_dict[t_prot]=optimization_result

    min_heats.append(optimization_result.fun)

    print("The minimal heat was found to be : ",optimization_result.fun)

    print("The Jacobian was found to be : ",optimization_result.jac)

RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =          200     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.00475D-01    |proj g|=  1.25076D-02


 This problem is unconstrained.



At iterate    1    f=  6.01116D-02    |proj g|=  9.58600D-03

At iterate    2    f=  1.87557D-02    |proj g|=  5.09343D-03

At iterate    3    f=  5.52411D-03    |proj g|=  3.14895D-03

At iterate    4    f= -2.99025D-04    |proj g|=  1.93903D-03

At iterate    5    f= -2.50575D-03    |proj g|=  1.48306D-03

At iterate    6    f= -3.89243D-03    |proj g|=  1.22867D-03

At iterate    7    f= -5.28558D-03    |proj g|=  9.27261D-04

At iterate    8    f= -6.12999D-03    |proj g|=  1.02122D-03
The minimal heat was found to be : 
 -0.006812844807497302
The Jacobian was found to be :  [ 5.17432224e-04  3.57082619e-04 -4.90415575e-04 -2.02103597e-04
  5.07265016e-04  5.57655933e-04 -3.16304489e-05 -6.00921449e-04
 -6.53076391e-04 -2.59712903e-04  2.04102902e-04  4.43980201e-04
  3.85579315e-04  1.28868066e-04 -1.60975032e-04 -3.52298173e-04
 -3.95189982e-04 -3.14253457e-04 -1.71520367e-04 -2.94578737e-05
  7.05604803e-05  1.13413350e-04  1.04942068e-04  6.26174721e-05
  6.65240650e-06 -4.614

In [9]:
# processtensor_simplemodel
file_name1='optimization_simplemodel'
with open(file_name1,'wb') as f:
    dill.dump(optimization_result,f)